# Baybayin Legacy Training (Combined)

This notebook trains a legacy (tesstrain) Baybayin model by combining the single-character and word-level datasets that live in this repository. Run the cells sequentially on a machine with the Tesseract training tools installed.

## Prerequisites
- Tesseract 4.1+ built with the training utilities (`lstmtraining`, `unicharset_extractor`, `combine_lang_model`).
- This repository cloned locally with the datasets under `kaggle_dataset_corrected_full_clean/` and `word_dataset/`.
- (Optional) Pillow installed for converting non-TIFF samples; it ships with this project via the existing virtual environment.

In [1]:
import platform
import shutil
import subprocess
import sys
from pathlib import Path

print(f"Python {platform.python_version()} on {platform.platform()}")
for cmd in ['tesseract', 'lstmtraining', 'unicharset_extractor', 'combine_lang_model']:
    path = shutil.which(cmd)
    print(f"{cmd:25s}: {path or 'NOT FOUND'}")


Python 3.10.12 on Linux-6.6.87.2-microsoft-standard-WSL2-x86_64-with-glibc2.35
tesseract                : /usr/bin/tesseract
lstmtraining             : /usr/bin/lstmtraining
unicharset_extractor     : /usr/bin/unicharset_extractor
combine_lang_model       : /usr/bin/combine_lang_model


In [2]:
# --- Configuration ---
from pathlib import Path

PROJECT_ROOT = Path('..').resolve()
TESSTRAIN_DIR = PROJECT_ROOT / 'tesseract_training'
SINGLE_CHAR_DIR = PROJECT_ROOT / 'kaggle_dataset_corrected_full_clean'
WORD_DATASET_DIR = PROJECT_ROOT / 'word_dataset'

MODEL_NAME = 'baybayin_legacy_combined'
START_MODEL = 'eng'
MAX_ITERATIONS = 20000  # adjust as needed; roughly analogous to training epochs

COMBINED_SOURCE_DIR = PROJECT_ROOT / 'train' / f'{MODEL_NAME}_combined_gt_source'
LANGDATA_DIR = TESSTRAIN_DIR / 'langdata'
TESSDATA_PREFIX = TESSTRAIN_DIR / 'data'

CLEAN_COMBINED_DIR = True   # wipe the combined directory before rebuilding it
CONVERT_TO_TIFF = True      # convert non-TIFF images to TIFF for tesstrain
RESAVE_DPI = 300            # set to None to preserve original DPI metadata

print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'Combined dataset will be staged in: {COMBINED_SOURCE_DIR}')


PROJECT_ROOT: /home/kmbandillo/baybayin_project
Combined dataset will be staged in: /home/kmbandillo/baybayin_project/train/baybayin_legacy_combined_combined_gt_source


In [3]:
from collections import Counter
from pathlib import Path
from PIL import Image
import shutil

if not SINGLE_CHAR_DIR.exists():
    raise FileNotFoundError(f'SINGLE_CHAR_DIR not found: {SINGLE_CHAR_DIR}')
if not WORD_DATASET_DIR.exists():
    raise FileNotFoundError(f'WORD_DATASET_DIR not found: {WORD_DATASET_DIR}')

sources = [
    (SINGLE_CHAR_DIR, 'glyph'),
    (WORD_DATASET_DIR, 'word'),
]

if CLEAN_COMBINED_DIR and COMBINED_SOURCE_DIR.exists():
    shutil.rmtree(COMBINED_SOURCE_DIR)
COMBINED_SOURCE_DIR.mkdir(parents=True, exist_ok=True)

copied = Counter()
for src_dir, tag in sources:
    gt_files = sorted(src_dir.glob('*.gt.txt'))
    if not gt_files:
        print(f"warning: no .gt.txt files found in {src_dir}")
        continue
    for gt_file in gt_files:
        if gt_file.name.endswith('.gt.txt'):
            basename = gt_file.name[:-7]
        else:
            basename = gt_file.stem
        combined_stem = f"{tag}_{basename}"

        dest_gt = COMBINED_SOURCE_DIR / f"{combined_stem}.gt.txt"
        text = gt_file.read_text(encoding='utf-8-sig').strip()
        dest_gt.write_text(text + '\n', encoding='utf-8')

        box_file = gt_file.with_name(basename + '.box')
        if box_file.exists():
            shutil.copy2(box_file, COMBINED_SOURCE_DIR / f"{combined_stem}.box")

        image_path = None
        for ext in ('.tif', '.tiff', '.png', '.jpg', '.jpeg'):
            candidate = gt_file.with_name(basename + ext)
            if candidate.exists():
                image_path = candidate
                break
        if image_path is None:
            raise FileNotFoundError(f"No image found for {gt_file.name} in {src_dir}")

        if image_path.suffix.lower() in ('.tif', '.tiff') and not CONVERT_TO_TIFF:
            dest_image = COMBINED_SOURCE_DIR / f"{combined_stem}{image_path.suffix.lower()}"
            shutil.copy2(image_path, dest_image)
        else:
            dest_image = COMBINED_SOURCE_DIR / f"{combined_stem}.tif"
            if image_path.suffix.lower() in ('.tif', '.tiff'):
                shutil.copy2(image_path, dest_image)
            else:
                with Image.open(image_path) as img:
                    save_kwargs = {}
                    if RESAVE_DPI:
                        save_kwargs['dpi'] = (RESAVE_DPI, RESAVE_DPI)
                    img.save(dest_image, format='TIFF', compression='tiff_lzw', **save_kwargs)

        copied[tag] += 1

print('Combined dataset ready:')
for tag, count in copied.items():
    print(f"  {tag:>5}: {count} samples")
print(f"Total samples: {sum(copied.values())}")


Combined dataset ready:
  glyph: 59000 samples
   word: 700 samples
Total samples: 59700


In [5]:
import shutil

unique_sequences = set()
for gt_file in sorted(COMBINED_SOURCE_DIR.glob('*.gt.txt')):
    content = gt_file.read_text(encoding='utf-8').strip()
    if content:
        unique_sequences.add(content)

wordlist_path = PROJECT_ROOT / f'{MODEL_NAME}.wordlist'
with wordlist_path.open('w', encoding='utf-8') as f:
    for item in sorted(unique_sequences):
        f.write(item + '\n')

langdata_model_dir = LANGDATA_DIR / MODEL_NAME
langdata_model_dir.mkdir(parents=True, exist_ok=True)
shutil.copy2(wordlist_path, langdata_model_dir / f'{MODEL_NAME}.wordlist')

print(f'Wordlist entries: {len(unique_sequences)}')
print(f'Wordlist copied to {langdata_model_dir / (MODEL_NAME + ".wordlist")}')


Wordlist entries: 750
Wordlist copied to /home/kmbandillo/baybayin_project/tesseract_training/langdata/baybayin_legacy_combined/baybayin_legacy_combined.wordlist


In [8]:
import subprocess
import sys

prep_cmd = [
    sys.executable,
    str(PROJECT_ROOT / 'prepare_data.py'),
    '--source', str(COMBINED_SOURCE_DIR),
    '--base-dir', str(TESSTRAIN_DIR),
    '--model-name', MODEL_NAME,
    '--langdata-dir', str(LANGDATA_DIR),
    '--no-dpi',
]

print('Running:', ' '.join(str(x) for x in prep_cmd))
subprocess.run(prep_cmd, check=True)


Running: /usr/bin/python3 /home/kmbandillo/baybayin_project/prepare_data.py --source /home/kmbandillo/baybayin_project/train/baybayin_legacy_combined_combined_gt_source --base-dir /home/kmbandillo/baybayin_project/tesseract_training --model-name baybayin_legacy_combined --langdata-dir /home/kmbandillo/baybayin_project/tesseract_training/langdata --no-dpi
### Step 0: Configuration ###
 source_gt_dir=/home/kmbandillo/baybayin_project/train/baybayin_legacy_combined_combined_gt_source
 base_dir=/home/kmbandillo/baybayin_project/tesseract_training
 model_name=baybayin_legacy_combined
 langdata_dir=/home/kmbandillo/baybayin_project/tesseract_training/langdata

### Step 2: Sanitizing ground truth files ###
✅ Ground truth files sanitized and copied to: /home/kmbandillo/baybayin_project/train/baybayin_legacy_combined_combined_gt_source_clean
### Step 3: Creating Baybayin wordlist ###
 Using wordlist at /home/kmbandillo/baybayin_project/legacy_offline_training/filipino_latin.txt
✅ Baybayin wordl

CompletedProcess(args=['/usr/bin/python3', '/home/kmbandillo/baybayin_project/prepare_data.py', '--source', '/home/kmbandillo/baybayin_project/train/baybayin_legacy_combined_combined_gt_source', '--base-dir', '/home/kmbandillo/baybayin_project/tesseract_training', '--model-name', 'baybayin_legacy_combined', '--langdata-dir', '/home/kmbandillo/baybayin_project/tesseract_training/langdata', '--no-dpi'], returncode=0)

In [9]:
ground_truth_dir = TESSTRAIN_DIR / 'data' / f'{MODEL_NAME}-ground-truth'
files = sorted(ground_truth_dir.glob('*.gt.txt'))
print(f'Ground truth files staged: {len(files)}')
print(f'Sample files: {[p.name for p in files[:5]]}')


Ground truth files staged: 59700
Sample files: ['glyph_a_1.gt.txt', 'glyph_a_10.gt.txt', 'glyph_a_100.gt.txt', 'glyph_a_1000.gt.txt', 'glyph_a_101.gt.txt']


In [10]:
import os
import subprocess

env = os.environ.copy()
env['TESSDATA_PREFIX'] = str(TESSDATA_PREFIX)

training_cmd = [
    'make',
    'training',
    f'MODEL_NAME={MODEL_NAME}',
    f'START_MODEL={START_MODEL}',
    f'TESSDATA={TESSDATA_PREFIX}',
    f'MAX_ITERATIONS={MAX_ITERATIONS}',
]

print('Running:', ' '.join(str(x) for x in training_cmd))
subprocess.run(training_cmd, cwd=TESSTRAIN_DIR, env=env, check=True)


Running: make training MODEL_NAME=baybayin_legacy_combined START_MODEL=eng TESSDATA=/home/kmbandillo/baybayin_project/tesseract_training/data MAX_ITERATIONS=20000
You are using make version: 4.3
combine_tessdata -u /home/kmbandillo/baybayin_project/tesseract_training/data/eng.traineddata data/eng/baybayin_legacy_combined
Extracting tessdata components from /home/kmbandillo/baybayin_project/tesseract_training/data/eng.traineddata
Wrote data/eng/baybayin_legacy_combined.lstm
Wrote data/eng/baybayin_legacy_combined.lstm-punc-dawg
Wrote data/eng/baybayin_legacy_combined.lstm-word-dawg
Wrote data/eng/baybayin_legacy_combined.lstm-number-dawg
Wrote data/eng/baybayin_legacy_combined.lstm-unicharset
Wrote data/eng/baybayin_legacy_combined.lstm-recoder
Wrote data/eng/baybayin_legacy_combined.version


Version string:4.00.00alpha:eng:synth20170629:[1,36,0,1Ct3,3,16Mp3,3Lfys64Lfx96Lrx96Lfx512O1c1]
17:lstm:size=11689099, offset=192
18:lstm-punc-dawg:size=4322, offset=11689291
19:lstm-word-dawg:size=3694794, offset=11693613
20:lstm-number-dawg:size=4738, offset=15388407
21:lstm-unicharset:size=6360, offset=15393145
22:lstm-recoder:size=1012, offset=15399505
23:version:size=80, offset=15400517


unicharset_extractor --output_unicharset "data/baybayin_legacy_combined/my.unicharset" --norm_mode 2 "data/baybayin_legacy_combined/all-gt"
merge_unicharsets data/eng/baybayin_legacy_combined.lstm-unicharset data/baybayin_legacy_combined/my.unicharset "data/baybayin_legacy_combined/unicharset"
Loaded unicharset of size 112 from file data/eng/baybayin_legacy_combined.lstm-unicharset
Loaded unicharset of size 24 from file data/baybayin_legacy_combined/my.unicharset
Wrote unicharset file data/baybayin_legacy_combined/unicharset.


Bad box coordinates in boxfile string! ᜌ
Extracting unicharset from plain text file data/baybayin_legacy_combined/all-gt
Wrote unicharset file data/baybayin_legacy_combined/my.unicharset


PYTHONIOENCODING=utf-8 python3 generate_wordstr_box.py -i "data/baybayin_legacy_combined-ground-truth/glyph_ya_939.tif" -t "data/baybayin_legacy_combined-ground-truth/glyph_ya_939.gt.txt" > "data/baybayin_legacy_combined-ground-truth/glyph_ya_939.box"
tesseract "data/baybayin_legacy_combined-ground-truth/glyph_ya_939.tif" data/baybayin_legacy_combined-ground-truth/glyph_ya_939 --psm 10 lstm.train


...

tesseract "data/baybayin_legacy_combined-ground-truth/glyph_g_758.tif" data/baybayin_legacy_combined-ground-truth/glyph_g_758 --psm 10 lstm.train
PYTHONIOENCODING=utf-8 python3 generate_wordstr_box.py -i "data/baybayin_legacy_combined-ground-truth/glyph_e_i_585.tif" -t "data/baybayin_legacy_combined-ground-truth/glyph_e_i_585.gt.txt" > "data/baybayin_legacy_combined-ground-truth/glyph_e_i_585.box"
tesseract "data/baybayin_legacy_combined-ground-truth/glyph_e_i_585.tif" data/baybayin_legacy_combined-ground-truth/glyph_e_i_585 --psm 10 lstm.train
PYTHONIOENCODING=utf-8 python3 generate_wordstr_box.py -i "data/baybayin_legacy_combined-ground-truth/glyph_ye_yi_479.tif" -t "data/baybayin_legacy_combined-ground-truth/glyph_ye_yi_479.gt.txt" > "data/baybayin_legacy_combined-ground-truth/glyph_ye_yi_479.box"


Tesseract Open Source OCR Engine v4.1.1 with Leptonica
Page 1
Tesseract Open Source OCR Engine v4.1.1 with Leptonica
Page 1


tesseract "data/baybayin_legacy_combined-ground-truth/glyph_ye_yi_479.tif" data/baybayin_legacy_combined-ground-truth/glyph_ye_yi_479 --psm 10 lstm.train
PYTHONIOENCODING=utf-8 python3 generate_wordstr_box.py -i "data/baybayin_legacy_combined-ground-truth/word_Ituro.tif" -t "data/baybayin_legacy_combined-ground-truth/word_Ituro.gt.txt" > "data/baybayin_legacy_combined-ground-truth/word_Ituro.box"


Traceback (most recent call last):
  File "/home/kmbandillo/baybayin_project/tesseract_training/generate_wordstr_box.py", line 60, in <module>
    line = bidi.algorithm.get_display(line)
  File "/home/kmbandillo/.local/lib/python3.10/site-packages/bidi/algorithm.py", line 663, in get_display
    resolve_implicit_levels(storage, debug)
  File "/home/kmbandillo/.local/lib/python3.10/site-packages/bidi/algorithm.py", line 475, in resolve_implicit_levels
    assert _ch["type"] in ("L", "R", "EN", "AN"), (
AssertionError:  not allowed here
make: *** [Makefile:244: data/baybayin_legacy_combined-ground-truth/word_Ituro.box] Error 1


CalledProcessError: Command '['make', 'training', 'MODEL_NAME=baybayin_legacy_combined', 'START_MODEL=eng', 'TESSDATA=/home/kmbandillo/baybayin_project/tesseract_training/data', 'MAX_ITERATIONS=20000']' returned non-zero exit status 2.

In [ ]:
log_path = TESSTRAIN_DIR / 'data' / MODEL_NAME / 'training.log'
if log_path.exists():
    print(f'Last 20 lines of {log_path}:')
    for line in log_path.read_text(encoding='utf-8').splitlines()[-20:]:
        print(line)
else:
    print('training.log not found yet; training may still be running.')

traineddata_path = TESSTRAIN_DIR / 'data' / f'{MODEL_NAME}.traineddata'
if traineddata_path.exists():
    size_mb = traineddata_path.stat().st_size / (1024 * 1024)
    print(f'Final traineddata: {traineddata_path} ({size_mb:.2f} MiB)')
else:
    print('traineddata artefact not found yet.')


## Next Steps
- Adjust `MAX_ITERATIONS` (and other make variables such as `RATIO_TRAIN` or `NET_SPEC`) to experiment with different training durations.
- Use the evaluation scripts in `../evaluate_model.py` or `../generate_holdout_visuals.py` to benchmark the model on unseen samples.
- Archive the `tesseract_training/data` directory for reproducibility once you are happy with the trained model.
